# SalarySense — Data Science Salary Prediction (INR)
### Google Colab Notebook: Preprocessing, EDA, ML Model Training & Artifact Export

Welcome to **SalarySense**! This notebook builds a complete machine learning pipeline to predict Data Science job salaries, converted into **Indian Rupees (INR)** and **Lakhs Per Annum (LPA)** using a configurable exchange rate benchmark (₹85 / USD).

> **Currency Conversion Note**: The dataset contains international salaries originally normalized in USD. A benchmark exchange rate of **1 USD = 85.00 INR** is used throughout for illustrative salary comparisons in Indian numbering style (₹ and LPA). Note that currency conversion does not make international salaries equivalent to local Indian market salaries due to differences in purchasing power parity (PPP) and domestic hiring budgets.

## 1. Setup & Library Imports

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("All required libraries imported successfully.")

## 2. Configuration & Dataset Acquisition
We use the open **Data Science Job Salaries** dataset hosted on Hugging Face.

In [ ]:
# Configuration Constants
DATASET_URL = "https://huggingface.co/datasets/hugginglearners/data-science-job-salaries/resolve/main/ds_salaries.csv"
USD_TO_INR = 85.0  # Illustrative exchange rate: 1 USD = 85.00 INR
RANDOM_SEED = 42

print(f"Downloading dataset from: {DATASET_URL}")
df_raw = pd.read_csv(DATASET_URL)
print(f"Dataset loaded successfully. Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.")
df_raw.head()

## 3. Data Inspection & Diagnostics
Inspect data types, missing entries, duplicates, and column structures.

In [ ]:
print("=== Column Types and Non-Null Counts ===")
print(df_raw.info())

print("\n=== Missing Values Check ===")
missing_counts = df_raw.isnull().sum()
print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else "No missing values found.")

print("\n=== Duplicates Check ===")
duplicates_count = df_raw.duplicated().sum()
print(f"Exact duplicate rows found: {duplicates_count}")

print("\n=== Numerical Summary Statistics ===")
df_raw.describe()

## 4. Data Cleaning & Target Conversion (USD to INR)
- Drop index columns (`Unnamed: 0`).
- Remove duplicate records.
- Exclude redundant raw salary columns (`salary`, `salary_currency`) to prevent target leakage.
- Convert `salary_in_usd` to `salary_in_inr` and `salary_in_lpa` using the illustrative rate ₹85/USD.

In [ ]:
df = df_raw.copy()

# 1. Drop redundant index column
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# 2. Remove duplicate rows
prev_rows = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {prev_rows - len(df)} duplicate rows. Clean dataset size: {len(df)} rows.")

# 3. Compute INR Target Salary and LPA
df['salary_in_inr'] = df['salary_in_usd'] * USD_TO_INR
df['salary_in_lpa'] = df['salary_in_inr'] / 100000.0

# 4. Exclude redundant raw salary columns to prevent target leakage
redundant_cols = ['salary', 'salary_currency']
df = df.drop(columns=[col for col in redundant_cols if col in df.columns])

print("\nSample records with INR & LPA targets:")
df[['job_title', 'experience_level', 'employment_type', 'salary_in_usd', 'salary_in_inr', 'salary_in_lpa']].head()

## 5. Exploratory Data Analysis (EDA)
Visualizing the salary distribution, experience level progression, top job roles, and remote work arrangements in Indian Rupees.

In [ ]:
# Set aesthetic plotting styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Salary Distribution in LPA
sns.histplot(df['salary_in_lpa'], kde=True, color='#0D9488', ax=axes[0, 0])
median_lpa = df['salary_in_lpa'].median()
axes[0, 0].axvline(median_lpa, color='#E11D48', linestyle='--', label=f'Median: ₹{median_lpa:.1f} LPA')
axes[0, 0].set_title('Distribution of Data Science Salaries (₹ LPA at ₹85/USD)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Annual Salary (Lakhs Per Annum - ₹)')
axes[0, 0].set_ylabel('Number of Professionals')
axes[0, 0].legend()

# 2. Salary by Experience Level
exp_order = ['EN', 'MI', 'SE', 'EX']
exp_labels = ['Entry (EN)', 'Mid (MI)', 'Senior (SE)', 'Executive (EX)']
sns.boxplot(data=df, x='experience_level', y='salary_in_lpa', order=exp_order, palette='crest', ax=axes[0, 1])
axes[0, 1].set_xticklabels(exp_labels)
axes[0, 1].set_title('Salary Range by Experience Level (LPA)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Experience Level')
axes[0, 1].set_ylabel('Annual Salary (₹ LPA)')

# 3. Top 8 Job Titles by Median Salary
top_roles = df.groupby('job_title')['salary_in_lpa'].median().sort_values(ascending=False).head(8)
sns.barplot(x=top_roles.values, y=top_roles.index, palette='mako', ax=axes[1, 0])
axes[1, 0].set_title('Top 8 Highest Median Salaries by Job Role (LPA)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Median Salary (₹ LPA)')
axes[1, 0].set_ylabel('Job Title')

# 4. Salary by Remote Work Arrangement
remote_mapping = {0: 'On-site (0%)', 50: 'Hybrid (50%)', 100: 'Remote (100%)'}
df_remote = df.copy()
df_remote['remote_label'] = df_remote['remote_ratio'].map(remote_mapping)
sns.boxplot(data=df_remote, x='remote_label', y='salary_in_lpa', palette='viridis', ax=axes[1, 1])
axes[1, 1].set_title('Salary by Remote Work Arrangement', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Work Arrangement')
axes[1, 1].set_ylabel('Annual Salary (₹ LPA)')

plt.tight_layout()
plt.show()

## 6. Feature Selection & Preprocessing Pipeline
- Features: `work_year`, `remote_ratio`, `experience_level`, `employment_type`, `job_title`, `employee_residence`, `company_location`, `company_size`.
- Target: `salary_in_inr`.
- Preprocessing: `StandardScaler` for numerical features and `OneHotEncoder(handle_unknown='ignore')` for categorical features inside a unified `ColumnTransformer`.

In [ ]:
NUMERICAL_FEATURES = ['work_year', 'remote_ratio']
CATEGORICAL_FEATURES = [
    'experience_level',
    'employment_type',
    'job_title',
    'employee_residence',
    'company_location',
    'company_size'
]
FEATURE_COLS = NUMERICAL_FEATURES + CATEGORICAL_FEATURES
TARGET_COL = 'salary_in_inr'

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

print(f"Feature matrix columns ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Target column: {TARGET_COL}")

# Build ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUMERICAL_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES)
    ]
)
print("Preprocessor built successfully.")

## 7. Train-Test Split (Leakage-Free)
We split the data into 80% training and 20% testing sets using `random_state=42`. Preprocessing is fit strictly within pipelines on training folds.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED
)

print(f"Training set shape: {X_train.shape[0]} rows, {X_train.shape[1]} columns")
print(f"Testing set shape: {X_test.shape[0]} rows, {X_test.shape[1]} columns")

## 8. Model Training & 5-Fold Cross-Validation
We train and cross-validate four candidate regression algorithms:
1. **Linear Regression**
2. **Decision Tree Regressor**
3. **Random Forest Regressor**
4. **Gradient Boosting Regressor**

In [ ]:
candidate_models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=6, random_state=RANDOM_SEED),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=8, min_samples_split=4, random_state=RANDOM_SEED),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.08, max_depth=4, random_state=RANDOM_SEED)
}

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_results = []
fitted_pipelines = {}

print("Running 5-fold cross-validation on training data...")
for name, model in candidate_models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # Compute CV metrics
    r2_cv = cross_val_score(pipe, X_train, y_train, cv=kf, scoring='r2')
    neg_mae_cv = cross_val_score(pipe, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error')
    neg_rmse_cv = cross_val_score(pipe, X_train, y_train, cv=kf, scoring='neg_root_mean_squared_error')
    
    # Fit pipeline on entire train set
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe
    
    mean_r2 = float(r2_cv.mean())
    mean_mae = float(-neg_mae_cv.mean())
    mean_rmse = float(-neg_rmse_cv.mean())
    
    cv_results.append({
        'Model': name,
        'CV R²': round(mean_r2, 4),
        'CV MAE (₹)': round(mean_mae, 2),
        'CV MAE (LPA)': round(mean_mae / 100000.0, 2),
        'CV RMSE (₹)': round(mean_rmse, 2),
        'CV RMSE (LPA)': round(mean_rmse / 100000.0, 2)
    })

cv_df = pd.DataFrame(cv_results).sort_values(by='CV R²', ascending=False).reset_index(drop=True)
print("=== Cross-Validation Results (Training Set) ===")
cv_df

## 9. Evaluation on Held-Out Test Set
We evaluate each fitted pipeline on the untouched 20% test set using **MAE (₹ and LPA)**, **RMSE (₹ and LPA)**, and **R²**.

In [ ]:
test_metrics = []
test_predictions = {}

for name, pipe in fitted_pipelines.items():
    y_pred = pipe.predict(X_test)
    test_predictions[name] = y_pred
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    test_metrics.append({
        'Model': name,
        'Test R²': round(float(r2), 4),
        'Test MAE (₹)': round(float(mae), 2),
        'Test MAE (LPA)': round(float(mae / 100000.0), 2),
        'Test RMSE (₹)': round(float(rmse), 2),
        'Test RMSE (LPA)': round(float(rmse / 100000.0), 2)
    })

test_df = pd.DataFrame(test_metrics).sort_values(by='Test R²', ascending=False).reset_index(drop=True)
print("=== Held-Out Test Set Evaluation ===")
test_df

## 10. Model Comparison & Actual vs. Predicted Visualizations
Select the best model based on held-out test R² and error metrics, and visualize performance.

In [ ]:
best_model_name = test_df.iloc[0]['Model']
best_pipeline = fitted_pipelines[best_model_name]
best_test_preds = test_predictions[best_model_name]

print(f"Selected Champion Model: {best_model_name} (Test R² = {test_df.iloc[0]['Test R²']})")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Model Comparison on Test R2
sns.barplot(data=test_df, x='Model', y='Test R²', palette='mako', ax=axes[0])
axes[0].set_title('Test R² Score Comparison Across Regressors', fontsize=12, fontweight='bold')
axes[0].set_ylabel('R² Score')
axes[0].set_ylim(0, max(test_df['Test R²']) * 1.3)
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2., p.get_height() + 0.01), ha='center')

# 2. Actual vs Predicted in LPA for Best Model
y_test_lpa = y_test / 100000.0
best_preds_lpa = best_test_preds / 100000.0

axes[1].scatter(y_test_lpa, best_preds_lpa, color='#0F766E', alpha=0.7, edgecolors='w', s=65, label='Test Predictions')
min_val = min(y_test_lpa.min(), best_preds_lpa.min())
max_val = max(y_test_lpa.max(), best_preds_lpa.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction (y = x)')
axes[1].set_title(f'Actual vs. Predicted Salary ({best_model_name})', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Actual Salary (₹ LPA)')
axes[1].set_ylabel('Predicted Salary (₹ LPA)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 11. Export Model Artifacts (`salary_model.joblib` & `model_metrics.json`)
Export the complete fitted pipeline and metadata JSON for seamless loading in the Streamlit application.

In [ ]:
MODEL_FILE = "salary_model.joblib"
METRICS_FILE = "model_metrics.json"

# 1. Save complete fitted Scikit-Learn pipeline
joblib.dump(best_pipeline, MODEL_FILE)
print(f"Successfully exported fitted pipeline: {MODEL_FILE}")

# 2. Extract unique categorical options for dynamic Streamlit inputs
categorical_options = {}
for col in CATEGORICAL_FEATURES:
    categorical_options[col] = sorted(df[col].dropna().unique().tolist())

# 3. Prepare test sample comparisons for interactive visual inspection
samples_data = []
for actual, pred in zip(y_test.values, best_test_preds):
    samples_data.append({
        'actual_inr': round(float(actual), 2),
        'predicted_inr': round(float(pred), 2),
        'actual_lpa': round(float(actual / 100000.0), 2),
        'predicted_lpa': round(float(pred / 100000.0), 2),
        'error_lpa': round(float((pred - actual) / 100000.0), 2)
    })

# 4. Build comprehensive metadata dictionary
model_metadata = {
    "project_name": "SalarySense",
    "target_currency": "INR",
    "exchange_rate_usd_to_inr": USD_TO_INR,
    "exchange_rate_label": "1 USD = ₹85.00 INR (Illustrative Benchmark)",
    "disclaimer": "Currency conversion does not make international salaries equivalent to Indian domestic market salaries.",
    "best_model_name": best_model_name,
    "dataset_stats": {
        "total_records": int(len(df)),
        "median_salary_inr": round(float(df['salary_in_inr'].median()), 2),
        "median_salary_lpa": round(float(df['salary_in_lpa'].median()), 2),
        "mean_salary_inr": round(float(df['salary_in_inr'].mean()), 2),
        "mean_salary_lpa": round(float(df['salary_in_lpa'].mean()), 2),
        "min_salary_inr": round(float(df['salary_in_inr'].min()), 2),
        "min_salary_lpa": round(float(df['salary_in_lpa'].min()), 2),
        "max_salary_inr": round(float(df['salary_in_inr'].max()), 2),
        "max_salary_lpa": round(float(df['salary_in_lpa'].max()), 2)
    },
    "best_model_metrics": test_df[test_df['Model'] == best_model_name].to_dict(orient='records')[0],
    "model_comparison": test_df.to_dict(orient='records'),
    "features": {
        "numerical": NUMERICAL_FEATURES,
        "categorical": CATEGORICAL_FEATURES,
        "categorical_options": categorical_options
    },
    "actual_vs_predicted_samples": samples_data
}

with open(METRICS_FILE, 'w', encoding='utf-8') as f:
    json.dump(model_metadata, f, indent=2)

print(f"Successfully exported metadata and metrics: {METRICS_FILE}")

## 12. Download Instructions for Google Colab
Execute the cell below to trigger immediate downloads of both exported files in Google Colab.

In [ ]:
try:
    from google.colab import files
    print("Triggering browser download in Google Colab...")
    files.download('salary_model.joblib')
    files.download('model_metrics.json')
    print("Downloads triggered successfully! Place both files beside app.py in your SalarySense/ folder.")
except ImportError:
    print("Running locally. Artifacts are saved in the current working directory:")
    print(f"- {os.path.abspath('salary_model.joblib')}")
    print(f"- {os.path.abspath('model_metrics.json')}")